<a href="https://colab.research.google.com/github/Manoel-Moreira/Analise-Exploratoria_com_Pandas/blob/main/Projeto_Analise_Sentimento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# <font color='blue'>Fundamentos de Linguagem Python - Do Básico a Aplicações de IA</font>

# <font color='blue'>Modelo de Classificação Para Análise de Sentimentos</font>

## 1. Definição do Problema de Negócio

É a fundação do projeto. Nesta etapa, traduzimos uma necessidade da empresa em um objetivo claro para a Ciência de Dados. Definimos o que queremos resolver (automatizar a classificação de reviews de produtos, por exemplo), por que é importante (reduzir custos, agilizar a tomada de decisão) e como o sucesso será medido.

**Definição:**

Uma empresa de e-commerce deseja automatizar a análise de feedback de seus clientes. Atualmente, a análise é feita manualmente, o que é um processo lento, caro e que não escala com o volume de reviews recebidos diariamente.

**Objetivo:**

Criar um modelo de Machine Learning que classifique automaticamente os reviews de produtos como **'positivo'** ou **'negativo'**.

**Benefícios Esperados:**
* **Eficiência:** Reduzir o tempo e o custo da análise de feedback.
* **Tomada de Decisão Rápida:** Permitir que as equipes de produto e marketing identifiquem rapidamente produtos com problemas ou oportunidades de melhoria.
* **Priorização:** Direcionar reviews negativos para a equipe de suporte ao cliente de forma prioritária, melhorando a experiência do consumidor.

## 2. Importação dos pacotes

In [ ]:
# Instala o pacote watermark
!pip install -q -U watermark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.2 MB/s eta 0:00:00


In [ ]:
# Manipulação de dados e visualização
# Manipulação de dados e visualização
import re
import pandas as pd
import numpy as np
import unicodedata
import seaborn as sns
import matplotlib.pyplot as plt

# Pré-Processamento e Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

## Explicando cada importação
-- Manipulação de dados e visualização

import re  # Expressões regulares: busca/substituição e limpeza de texto (padronização, extração de padrões)

import pandas as pd  # pandas: estruturas de dados (DataFrame/Series) e ETL (ler CSV/Excel, agrupar, mesclar)

import numpy as np  # NumPy: arrays eficientes, operações vetorizadas e funções numéricas

import unicodedata  # Normalização Unicode: remover acentos e padronizar caracteres

import seaborn as sns  # Seaborn: visualizações estatísticas de alto nível com temas prontos

import matplotlib.pyplot as plt  # Matplotlib: base de gráficos (customização detalhada e plots básicos)

-- Pré-Processamento e Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV  # train_test_split: separa treino/teste; GridSearchCV: busca de hiperparâmetros com validação cruzada

from sklearn.preprocessing import StandardScaler  # StandardScaler: padroniza features (média=0, desvio=1)

from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF: transforma texto em vetores ponderados por frequência e relevância

from sklearn.linear_model import LogisticRegression  # Regressão Logística: classificação linear (binária/multiclasse)

from sklearn.pipeline import Pipeline  # Pipeline: encadeia etapas (ex.: scaler -> modelo) evitando vazamento de dados

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix  # Métricas: acurácia, relatório (prec/recall/F1) e matriz de confusão

import joblib  # joblib: salvar/carregar modelos e objetos grandes (serialização eficiente)

In [ ]:
# Configurações de visualização
sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
%reload_ext watermark
%watermark -a "Data Science Academy"

Author: Data Science Academy



In [ ]:
%watermark --iversions

joblib    : 1.5.3
matplotlib: 3.10.0
numpy     : 2.0.2
pandas    : 2.2.2
re        : 2.2.1
seaborn   : 0.13.2
sklearn   : 1.6.1



### 3. Carregando a base de dados csv


In [ ]:
# Como o arquivo está salvo no meu drive, vou importar ele diretamente de lá
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Agora, vamos usar o ID do arquivo do Google Drive para construir um link de download direto. O ID do seu arquivo é `1Cbjn5BrjzscUAPkO9b2vcvcXKV5dtp_U`. Substitua este ID caso você queira carregar outro arquivo no futuro.

**Importante:** Certifique-se de que o arquivo esteja configurado para compartilhamento como 'Qualquer pessoa com o link pode visualizar' para que o Colab consiga acessá-lo.

In [ ]:
# ID do arquivo do Google Drive extraído do seu link original
file_id = '1Cbjn5BrjzscUAPkO9b2vcvcXKV5dtp_U'

# Constrói o link de download direto
file_path = f'https://drive.google.com/uc?export=download&id={file_id}'

# Carrega o arquivo CSV usando pandas
df = pd.read_csv(file_path)

# Exibe as primeiras 5 linhas do DataFrame para verificar
display(df.head())

,review_id,texto_review,sentimento
0,1,Estou muito feliz com a compra. O cadeira game...,positivo
1,2,NaN,negativo
2,3,Não recomendo. A entrega foi lenta e o celular...,negativo
3,4,O monitor é decepcionante. O suporte ao client...,positivo
4,5,É UM LIVRO OK PELO PRÇEO QUE PAGUEI.,negativo


In [ ]:
# Shape exibe a quantidade de linhas e colunas do dataframe (df)
df.shape

(500, 3)

In [ ]:
# sample exibe dados aleatórios
df.sample(10)

,review_id,texto_review,sentimento
402,403,O cadeira gamer é exatamente como descrito no ...,negativo
61,62,Estou muito feliz com a compra. O cadeira game...,positivo
343,344,O livro é exatamente como descrito no anúncio....,negativo
406,407,Estou muito frustrado com esta compra. Dinheir...,negativo
275,276,Amei o celular! A qualidade é incrível e a enr...,positivo
254,255,Péssima experiência. O celular quebrou no prmi...,negativo
100,101,É um teclado ok pelo preço que paguei.,negativo
450,451,Odiei o livro. Qualidade decepcionante e veio ...,negativo
461,462,Amei o teclado! A qualidade é fantástica e a e...,positivo
396,397,Ótimo custo-benefício. O mouse é fantástica e ...,positivo


In [ ]:
# Exibe as 5 últimas linhas
df.tail()

,review_id,texto_review,sentimento
495,496,Odiei o teclado. Qualidade de baixa qualidade ...,negativo
496,497,Estou muito impressionado com a compra. O moni...,positivo
497,498,Não recomendo. A entrega demorou uma eternidad...,negativo
498,499,Estou muito arrependido com esta compra. Dinhe...,negativo
499,500,Ótimo custo-benefício. O cadeira gamer é incrí...,positivo
